# Notebook 01: Reproduce RQ1 matrices and Table 1

This notebook reproduces Figure 2 (the EM x SW agreement matrices for the
three campaigns) and Table 1 (the per-campaign certainty rate R_certain
with its strict/loose split) of the paper.

Inputs: ``../data/rq1_agreement_matrices.csv`` and ``../data/rq1_em_evidence.csv``.

The strict/loose split per individual input cannot be recovered from the
marginal cell counts alone. The values shown in Table 1 are taken from
the per-input agreement records of the proprietary pipeline. The marginal
CSV is sufficient to verify cell sums, diagonal totals, and the matrix
visualisation. See the docstring of ``scripts/compute_certainty_rates.py``
for a longer explanation.


In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

LABELS = ["CPU", "Memory", "IO", "Clk"]
TRIAGE = {"CPU", "Memory", "IO"}
CAMPAIGNS = ("C2", "C3", "picoc")
N_INPUTS = {"C2": 14, "C3": 20, "picoc": 57}

DATA_DIR = Path("..") / "data"


## Load matrices


In [ ]:
def load_matrices(csv_path: Path) -> dict:
    matrices = {camp: np.zeros((4, 4), dtype=int) for camp in CAMPAIGNS}
    label_index = {label: idx for idx, label in enumerate(LABELS)}
    with csv_path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            camp = row["campaign"]
            i = label_index[row["em_hypothesis"]]
            j = label_index[row["sw_hypothesis"]]
            matrices[camp][i, j] = int(row["count"])
    return matrices

matrices = load_matrices(DATA_DIR / "rq1_agreement_matrices.csv")
for camp in CAMPAIGNS:
    print(f"{camp}: {int(matrices[camp].sum())}")


Expected output: ``C2: 11``, ``C3: 30``, ``picoc: 69``.


## Plot matrices

Cells eligible for ``certain`` attribution are highlighted with a thick red
border. A cell is eligible when its EM label equals its SW label and the
label is in {CPU, Memory, IO}. The Clk diagonal cell is not highlighted.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))
vmax = max(int(m.max()) for m in matrices.values())

for ax, camp in zip(axes, CAMPAIGNS):
    m = matrices[camp]
    ax.imshow(m, cmap="Blues", vmin=0, vmax=vmax)
    for i, em_label in enumerate(LABELS):
        for j, sw_label in enumerate(LABELS):
            value = int(m[i, j])
            colour = "white" if value > vmax * 0.55 else "black"
            ax.text(j, i, str(value), ha="center", va="center", color=colour)
            if em_label == sw_label and em_label in TRIAGE:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1.0, 1.0,
                                        fill=False, edgecolor="red", linewidth=2.5))
    ax.set_xticks(range(len(LABELS)))
    ax.set_yticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS)
    ax.set_yticklabels(LABELS)
    ax.set_xlabel("Software hypothesis")
    ax.set_ylabel("EM hypothesis")
    ax.set_title(f"{camp}\n(unique inputs: {N_INPUTS[camp]}, cell sum: {int(m.sum())})")

fig.tight_layout()
plt.show()


## Reproduce Table 1


In [ ]:
PAPER_TABLE1 = {
    "C2":    {"N_A": 14, "R_certain": 0.071, "strict_share": 0.000, "loose_share": 1.000},
    "C3":    {"N_A": 20, "R_certain": 0.500, "strict_share": 0.000, "loose_share": 1.000},
    "picoc": {"N_A": 57, "R_certain": 0.246, "strict_share": 0.643, "loose_share": 0.357},
}

header = f"{'Campaign':<10}{'N_A':>6}{'R_certain':>12}{'Strict%':>10}{'Loose%':>10}"
print(header)
print("-" * len(header))
for camp in CAMPAIGNS:
    info = PAPER_TABLE1[camp]
    print(f"{camp:<10}{info['N_A']:>6}{info['R_certain']:>12.3f}"
          f"{info['strict_share']*100:>9.1f}%{info['loose_share']*100:>9.1f}%")


## EM evidence distribution

The directional percentage per campaign is defined as the share of inputs
whose EM evidence falls into one of the three directional bands
(High, Mid, Low), as opposed to the Weak band (no directional evidence)
or the Clock band (a bookkeeping label). It is computed as
``directional_pct = 100 - (Weak% + Clock%)``.


In [ ]:
evidence: dict = {camp: {} for camp in CAMPAIGNS}
with (DATA_DIR / "rq1_em_evidence.csv").open(newline="") as handle:
    for row in csv.DictReader(handle):
        evidence[row["campaign"]][row["band"]] = float(row["percentage"])

print(f"{'Campaign':<10}{'Directional%':>15}")
print("-" * 25)
for camp in CAMPAIGNS:
    weak = evidence[camp].get("Weak", 0.0)
    clock = evidence[camp].get("Clock", 0.0)
    directional = 100.0 - weak - clock
    print(f"{camp:<10}{directional:>14.1f}%")


Expected: C2 = 22.2%, C3 = 100.0%, picoc = 40.5%.

This directional percentage tracks ``R_certain`` across campaigns: when
a larger fraction of the EM evidence is directional, more inputs become
eligible for cross-modal agreement, and ``R_certain`` rises.
